# Chapter 4 &mdash; Caveat: Failing to Falsify Proves Nothing

**Concept 19 of the Chapter 4 decomposition:** *Caveat: Failing to Falsify $Cond$ Proves Nothing*

A non-regular language may still resist this lemma. Failure means the tool did not settle it.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Failure-Proves-Nothing/Concept-Failure-Proves-Nothing.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


You can apply the Pumping Lemma systematically to a **genuinely non-regular** language
and still fail to falsify $Cond$.

Does that make it regular? **No.**

The lemma is *sufficient*, not *complete*. So the implication is one-way in **both**
directions of use:

* satisfying $Cond$ does not make a language regular;
* failing to refute $Cond$ does not either.

Failure means only: **this tool did not settle the question.**

## 2. Definitions

### A language that resists

$L_{if} = \{a^ib^jc^k : \text{if } i=3 \text{ then } j=k\}$ &mdash; non-regular, but slippery.

In [ ]:
def in_Lif(s):
    i = len(s) - len(s.lstrip('a')); rest = s[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    if rest[j:] != 'c'*k: return False
    if s != 'a'*i + 'b'*j + 'c'*k: return False
    return (j == k) if i == 3 else True

### Try to pump it

In [ ]:
def try_pump(in_L, w, N, imax=3):
    surviving = []
    for p in range(N+1):
        for q in range(p+1, min(N, len(w))+1):
            x, y, z = w[:p], w[p:q], w[q:]
            if all(in_L(x + y*i + z) for i in range(imax+1)):
                surviving.append((x, y, z))
    return surviving

<!-- nav-strip -->

---

&larr;&nbsp;[Ch4&nbsp;18.&nbsp;Worked Proof: $L_{01}=\{0^i1^i\}$ is Not Regular](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-L01-Not-Regular/Concept-L01-Not-Regular.ipynb) &nbsp;&middot;&nbsp; [**Chapter 4** index](https://github.com/ganeshutah/Jove/blob/master/Chapter4-DFA/README.md) &nbsp;&middot;&nbsp; [Ch4&nbsp;20.&nbsp;Grossly Abusing the Pumping Lemma: the Language $L_{if}$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Abusing-The-Pumping-Lemma/Concept-Abusing-The-Pumping-Lemma.ipynb)&nbsp;&rarr;

---

## 3. Tests

The language is as advertised.

In [ ]:
for s in ['aaabbcc', 'aaabbbccc', 'aaa', 'aabbbccc', 'aaabcc', 'aaab']:
    print("%-12r in L_if ? %s" % (s, in_Lif(s)))
assert in_Lif('aaabbcc') and not in_Lif('aaabcc')

Pumping $w = aaa\,b^N c^N$ leaves a **surviving split** &mdash; the proof fails.

In [ ]:
N = 4; w = 'aaa' + 'b'*N + 'c'*N
surv = try_pump(in_Lif, w, N)
print("w =", w)
print("surviving splits :", surv[:4], "..." if len(surv) > 4 else "")
print("count            :", len(surv))
assert surv, "some split survives -- that is the point"
print("\nA y inside the a's changes the a-count away from 3, which RELEASES")
print("the j=k obligation. The pumped string stays in L_if.")

And yet $L_{if}$ **is** non-regular. Failure of the tool is not evidence.

In [ ]:
print("surviving split found  -> this lemma did not settle it")
print("L_if regular?          -> NO (Chapter 4 proves it by reversal, Concept 21)")
print()
print("'I tried and could not break it' is NOT a proof of regularity.")

## 4. Exercises


1. Which split survives, and why does changing the $a$-count help it?
2. Name two things to try when the lemma fails. (Concepts 21 and 22.)
3. Is there a language where *no* version of the lemma works? (Look up Jaffe.)

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4-DFA/Concept-Failure-Proves-Nothing')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')